# Blood Work Analysis Pipeline
A two-stage LLM pipeline:
- **Stage 1:** Extract and flag abnormal values from a blood report
- **Stage 2:** Generate a health summary and Indian diet plan based on flagged values

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

load_dotenv()

llm1=ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm2=ChatGroq(model="qwen/qwen3-32b")

D:\Ramvikas S V\projects\codebasics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("blood_work.txt","r") as f:
    blood_report=f.read()
print(blood_report[:200])

Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
--------------------------
Hemoglobin:        15.1 g/dL        (Normal: 13.5â€“17.5)
Hematocrit:        44%          


## Stage 1 — Extract and Flag Abnormal Values

In [7]:
extraction_prompt=f"""
You are a medical data extraction assistant.

From the blood report below, extract ALL test values and classify each one as HIGH, LOW, or NORMAL 
based on the reference ranges provided in the report.

Format your response as:
- Test Name: value | Status: HIGH/LOW/NORMAL | Reference: range

Blood Report:
{blood_report}
"""

extraction_response = llm1.invoke(extraction_prompt)
extracted_values = extraction_response.text

print("=== STAGE 1: EXTRACTED VALUES ===")
print(extracted_values)

=== STAGE 1: EXTRACTED VALUES ===
Here are the extracted test values and their classifications:

*   Hemoglobin: 15.1 g/dL | Status: NORMAL | Reference: 13.5–17.5
*   Hematocrit: 44% | Status: NORMAL | Reference: 41–53%
*   WBC: 6.8 x10^3/uL | Status: NORMAL | Reference: 4.5–11.0
*   Platelets: 220 x10^3/uL | Status: NORMAL | Reference: 150–400
*   Total Cholesterol: 238 mg/dL | Status: HIGH | Reference: <200
*   LDL Cholesterol: 162 mg/dL | Status: HIGH | Reference: <100
*   HDL Cholesterol: 36 mg/dL | Status: LOW | Reference: >40
*   Triglycerides: 188 mg/dL | Status: HIGH | Reference: <150
*   Glucose (Fasting): 92 mg/dL | Status: NORMAL | Reference: 70–99
*   HbA1c: 5.3% | Status: NORMAL | Reference: <5.7%
*   Creatinine: 1.0 mg/dL | Status: NORMAL | Reference: 0.7–1.3
*   eGFR: 82 mL/min | Status: NORMAL | Reference: >60
*   ALT: 28 U/L | Status: NORMAL | Reference: 7–40
*   AST: 25 U/L | Status: NORMAL | Reference: 10–40
*   Bilirubin Total: 0.8 mg/dL | Status: NORMAL | Reference

## Stage 2 — Health Summary and Indian Diet Plan

In [9]:
diet_prompt = f"""
You are a clinical nutritionist specializing in Indian dietary habits.

Based on the blood work analysis below, write:
1. A short health summary in 4-5 lines explaining the patient's condition in simple language
2. A short, practical Indian diet plan having only two sections (1) Foods to avoid (2) Foods to eat more of. 
   Do not include any other sections in diet plan.

Blood Work Analysis:
{extracted_values}
"""

diet_response = llm1.invoke(diet_prompt)

print("=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
print(diet_response.text)

=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===
Here's your health summary and diet plan:

### 1. Health Summary

Your blood tests show that your cholesterol levels are elevated, specifically your "bad" cholesterol (LDL) and triglycerides are high, while your "good" cholesterol (HDL) is a bit low. This indicates a higher risk for heart-related issues in the long run. Good news is, your blood sugar, kidney, and liver functions are all healthy. We need to focus on dietary changes to improve your heart health markers.

### 2. Practical Indian Diet Plan

**Foods to Avoid:**

*   **Deep-fried items:** Samosas, pakoras, puri, bhature, jalebi, commercially fried chips.
*   **Excessive Ghee & Butter:** Limit generous use of ghee and butter in cooking and as a topping.
*   **Full-Fat Dairy Products:** Avoid full-fat milk, malai (cream), and full-fat paneer.
*   **Red & Processed Meats:** Limit mutton, beef, and processed meats like sausages or cold cuts.
*   **Sugary Foods:** Mithai (Indian sweets)

 ##single step llm calling

In [12]:
prompt=f"""
you are an expert medical data extractor and nutrition specilist 

see the attached blood work report and check the value for all components and identify the normal, high, low and suggest the Indian home diet plan for the individual, just include 
3 line summary, foods to eat and foods to avoid, dont assume anything



Blood Report:
{blood_report}
"""

res=llm2.invoke(prompt,reasoning_format="parsed")
print(res.content)

**3-Line Summary:**  
Normal CBC and metabolic panel; elevated total/LDL cholesterol and low HDL indicate dyslipidemia. No diabetes or liver/kidney concerns. Focus on reducing unhealthy fats and increasing heart-healthy nutrition.  

**Foods to Eat:**  
- Whole grains (brown rice, oats, millet), legumes (toor dal, chana), leafy greens (spinach, fenugreek), fatty fish (salmon, mackerel), nuts (almonds, walnuts), avocados, and olive oil. Incorporate spices like turmeric and garlic.  
- Include soluble fiber-rich foods like oats, psyllium, and fruits (apples, pears).  

**Foods to Avoid:**  
- Processed meats (sausages, bacon), fried snacks (pakoras, samosas), refined oils (vanaspati, coconut oil), sugary desserts (gulab jamun, halwa), and full-fat dairy (paneer, cheese). Limit egg yolks and organ meats.
